# RandomizedSearchCV: Searching Large Hyperparameter Spaces Efficiently

## The Problem: Grid Search Doesn't Scale

GridSearchCV is rigorous and reproducible, but it doesn't scale.

**The Dimensionality Explosion:**
- 4 hyperparameters × 10 values each × 5-fold CV = **50,000 model fits**
- 5 hyperparameters × 10 values each × 5-fold CV = **500,000 model fits**

For complex models (Random Forests, Gradient Boosting, XGBoost), each fit takes seconds — and 500,000 fits can take **days or weeks**.

Grid search's exponential scaling makes it unsuitable for the models that need the most tuning.

## The Solution: Random Search

**RandomizedSearchCV** samples a fixed number of configurations at random from specified distributions — and finds **near-optimal solutions with a fraction of the compute**.

**Key Insight (Bergstra & Bengio, 2012):**  
When only a subset of hyperparameters significantly drives performance, random search explores the important dimensions far more efficiently than grid search.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import randint, uniform, loguniform
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All required libraries imported successfully")

## Section 1: Why Grid Search Becomes Inefficient

### The Core Problem: Wasted Computation

Grid search wastes computation **uniformly** across the search space — even on dimensions that barely affect performance.

**Example:** Random Forest searching over:
- `max_depth` [2, 4, 6, 8, ...] — highly sensitive to performance
- `min_samples_split` [2, 5, 10, 15, ...] — nearly insensitive within reasonable range
- `min_samples_leaf` [1, 3, 5, 7, ...] — nearly insensitive within reasonable range

Grid search applies the **same effort** to each dimension, burning through compute budget on dimensions that contribute almost nothing to performance variation.

### The Dimensionality Problem

Grid combinations scale **multiplicatively**:
- 3 hyperparameters × 10 values = 1,000 combinations
- 4 hyperparameters × 10 values = 10,000 combinations  
- 5 hyperparameters × 10 values = 100,000 combinations

Every new hyperparameter multiplies the cost. This **combinatorial explosion** makes exhaustive search impractical.

In [ ]:
# Demonstrate the combinatorial explosion
dimensions = [1, 2, 3, 4, 5, 6]
values_per_dim = 10
cv_folds = 5

grid_combinations = [np.prod([values_per_dim] * d) for d in dimensions]
total_fits = [comb * cv_folds for comb in grid_combinations]

results_explosion = pd.DataFrame({
    'Hyperparameters': dimensions,
    'Grid Combinations': grid_combinations,
    '5-Fold CV Fits': total_fits
})

print("The Combinatorial Explosion (10 values per dimension, 5-fold CV)")
print("="*70)
print(results_explosion.to_string(index=False))

print("\n" + "="*70)
print("\nObservation:")
for i in range(1, len(dimensions)):
    ratio = total_fits[i] / total_fits[i-1]
    print(f"  {dimensions[i]} dims vs {dimensions[i-1]} dims: {ratio:.0f}× increase")

print("\n✓ GridSearchCV's cost grows **exponentially** with dimensions")
print("✓ RandomizedSearchCV's cost scales **linearly** with n_iter (independent of dimensions)")

## Section 2: The Key Insight Behind Random Search

### The Geometric Intuition

**If only 2 of 5 hyperparameters significantly affect performance, then:**

| Approach | Grid Search (10 values each) | Random Search (50 iterations) |
|---|---|---|
| Total configurations evaluated | 10^5 = 100,000 | 50 |
| Unique looks at important dimension 1 | 10 | 50 ✓ |
| Unique looks at important dimension 2 | 10 | 50 ✓ |
| Unique looks at unimportant dimensions | 10 (wasted effort) | Varies (efficient sampling) |

**Random search gets 50 distinct "looks" at each important dimension independently**, while grid search gets only 10 — even though both test roughly the same total number of configurations.

### Why This Matters

In practice:
- **Most hyperparameters have limited impact** on model performance
- **A few hyperparameters drive most of the variation**
- Random search explores important dimensions far more thoroughly than grid search for the same computational budget
- Random search typically finds configurations within **a few percent** of the true optimum

This is why Bergstra & Bengio (2012) concluded that random search is preferable to grid search in high-dimensional hyperparameter spaces.

## Section 3: Defining Parameter Distributions

### Why Distributions Matter

The distribution assigned to each hyperparameter determines what values RandomizedSearchCV can sample. **Choosing the right distribution is as important as defining the right range.**

### Distribution Types

#### 1. **randint** — Uniform Discrete (Integers)

Draws integers uniformly from `[low, high)` with equal probability for each integer.

```python
from scipy.stats import randint
"n_estimators": randint(50, 500)    # [50, 500) uniformly
"max_depth": randint(2, 30)         # [2, 30) uniformly
```

**Use for:** n_estimators, max_depth, min_samples_split, min_samples_leaf, K in KNN

#### 2. **uniform** — Uniform Continuous

Draws real numbers uniformly from `[loc, loc + scale)` — all values equally likely.

```python
from scipy.stats import uniform
"learning_rate": uniform(0.01, 0.29)   # [0.01, 0.30) uniformly
```

**Use for:** Parameters where differences in magnitude are equally important across the range

#### 3. **loguniform** — Log-Uniform (Most Useful for Regularization)

Draws values uniformly in **log space**. Equivalent to sampling the exponent uniformly then exponentiating.

```python
from scipy.stats import loguniform
"C": loguniform(1e-4, 1e2)   # [0.0001, 100] log-uniformly
```

**Why loguniform is essential:**
- Change from 0.001 to 0.01 (one order of magnitude) ≈ Change from 10 to 100 (another order of magnitude)
- With `uniform(1e-4, 1e2)`: almost all samples are > 1 (ignores 0.0001–1)
- With `loguniform(1e-4, 1e2)`: all orders of magnitude get equal attention

**Use for:** C in Logistic Regression/SVM, alpha in Ridge/Lasso, learning rates in Gradient Boosting

#### 4. **Lists** — Categorical Choices

```python
"max_features": ["sqrt", "log2", None]   # Uniform sampling from three options
```

**Use for:** Algorithm-selection parameters without continuous interpretation

In [ ]:
# Create a sample dataset for demonstrations
X, y = make_classification(
    n_samples=1000, 
    n_features=20, 
    n_informative=15,
    n_redundant=5,
    random_state=42,
    class_sep=1.0
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.datasets import make_classification

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size:     {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")

# Visualize the difference between uniform and loguniform
n_samples = 1000

uniform_samples = np.random.uniform(0.0001, 100, n_samples)
loguniform_samples = loguniform(1e-4, 1e2).rvs(n_samples, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Histogram - Uniform
axes[0, 0].hist(uniform_samples, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Parameter Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('uniform(0.0001, 100) - Linear Scale')
axes[0, 0].grid(True, alpha=0.3)

# Histogram - LogUniform
axes[0, 1].hist(loguniform_samples, bins=50, color='darkorange', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Parameter Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('loguniform(1e-4, 1e2) - Linear Scale')
axes[0, 1].grid(True, alpha=0.3)

# Log scale histogram - Uniform
axes[1, 0].hist(uniform_samples, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[1, 0].set_xscale('log')
axes[1, 0].set_xlabel('Parameter Value (log scale)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('uniform(0.0001, 100) - Log Scale\n(Notice concentration near 100)')
axes[1, 0].grid(True, alpha=0.3, which='both')

# Log scale histogram - LogUniform
axes[1, 1].hist(loguniform_samples, bins=50, color='darkorange', alpha=0.7, edgecolor='black')
axes[1, 1].set_xscale('log')
axes[1, 1].set_xlabel('Parameter Value (log scale)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('loguniform(1e-4, 1e2) - Log Scale\n(Uniform distribution per order of magnitude)')
axes[1, 1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# Statistics
print("\n" + "="*70)
print("Distribution Comparison: uniform vs. loguniform")
print("="*70)
print(f"\nuniform(0.0001, 100):")
print(f"  Mean: {uniform_samples.mean():.2f}")
print(f"  Median: {np.median(uniform_samples):.2f}")
print(f"  < 1: {(uniform_samples < 1).sum() / len(uniform_samples) * 100:.1f}% of samples")
print(f"  >= 1: {(uniform_samples >= 1).sum() / len(uniform_samples) * 100:.1f}% of samples")

print(f"\nloguniform(1e-4, 1e2):")
print(f"  Mean: {loguniform_samples.mean():.2f}")
print(f"  Median: {np.median(loguniform_samples):.2f}")
print(f"  < 1: {(loguniform_samples < 1).sum() / len(loguniform_samples) * 100:.1f}% of samples")
print(f"  >= 1: {(loguniform_samples >= 1).sum() / len(loguniform_samples) * 100:.1f}% of samples")

print("\n✓ loguniform ensures all orders of magnitude receive equal attention")

## Section 4: Implementing RandomizedSearchCV with Random Forest

### Basic Implementation

RandomizedSearchCV samples `n_iter` random configurations from the parameter distributions and evaluates each with cross-validation.

In [ ]:
# Define parameter distributions for Random Forest
param_dist_rf = {
    "n_estimators":     randint(50, 500),       # [50, 500) uniformly
    "max_depth":        randint(2, 30),         # [2, 30) uniformly
    "min_samples_split": randint(2, 20),        # [2, 20) uniformly
    "min_samples_leaf":  randint(1, 15),        # [1, 15) uniformly
    "max_features":     ["sqrt", "log2", None]  # Categorical
}

print("Random Forest Parameter Distributions")
print("="*80)
print("""
├─ n_estimators:      randint(50, 500)          [50–500 trees]
├─ max_depth:         randint(2, 30)            [2–30 tree depth]
├─ min_samples_split: randint(2, 20)            [2–20 samples to split]
├─ min_samples_leaf:  randint(1, 15)            [1–15 samples per leaf]
└─ max_features:      ['sqrt', 'log2', None]    [Feature selection strategy]
""")

# If this were a grid search:
print("If this were GridSearchCV:")
grid_combinations_rf = 1
for key, value in param_dist_rf.items():
    if isinstance(value, list):
        grid_combinations_rf *= len(value)
    elif hasattr(value, '_pdf'):
        # It's a scipy distribution object
        grid_combinations_rf *= np.inf
        
print(f"  With RandomizedSearchCV n_iter=50: 50 configurations evaluated")
print(f"  With GridSearchCV on same space: 450 × 28 × 18 × 14 × 3 = ~4.8M combinations!")
print(f"  Speedup: ~96,000×\n")

# Run RandomizedSearchCV
print("Running RandomizedSearchCV with n_iter=50...")
print("(This takes ~30-60 seconds)\n")

rf = RandomForestClassifier(random_state=42)

random_search_rf = RandomizedSearchCV(
    rf,
    param_distributions=param_dist_rf,
    n_iter=50,               # Evaluate 50 random configurations
    cv=5,
    scoring="f1_weighted",
    return_train_score=True,
    random_state=42,         # ✓ CRITICAL for reproducibility
    n_jobs=-1                # Use all CPU cores
)

random_search_rf.fit(X_train, y_train)

print("✓ RandomizedSearchCV complete!")
print(f"\nBest Parameters:")
for param, value in random_search_rf.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV F1 Score: {random_search_rf.best_score_:.4f}")

# Evaluate on test set — exactly once, after tuning is complete
y_pred_rf = random_search_rf.best_estimator_.predict(X_test)
test_f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

print(f"Test F1 Score:   {test_f1_rf:.4f}")
print("\n✓ Note: random_state is set for reproducibility")

## Section 5: Determining the Right n_iter

### How Many Iterations Are Enough?

`n_iter` is the single most important parameter — it controls the exploration budget.

| Search Space Complexity | Recommended n_iter |
|---|---|
| 2–3 hyperparameters, narrow ranges | 20–30 |
| 3–5 hyperparameters, moderate ranges | 50–100 |
| 5+ hyperparameters, wide ranges | 100–200+ |
| Initial exploration / fast iteration | 20–30 |
| Final tuning for production | 100+ |

### empirical Approach: Plot CV Score vs. n_iter

The ideal approach: run RandomizedSearchCV for increasing values of n_iter and see when additional iterations stop producing meaningful improvement.

In [ ]:
# Determine optimal n_iter by seeing when CV scores plateau
n_iters_to_test = [10, 20, 30, 40, 50, 75, 100]
cv_scores_by_iter = []
timing_info = []

print("Finding the optimal n_iter...")
print("="*70)
print(f"{'n_iter':<10} {'Best CV Score':<15} {'Improvement':<15}")
print("-"*70)

import time

prev_score = 0
for n in n_iters_to_test:
    start_time = time.time()
    
    rs = RandomizedSearchCV(
        RandomForestClassifier(random_state=42),
        param_distributions=param_dist_rf,
        n_iter=n,
        cv=3,  # Use 3-fold for speed
        scoring="f1_weighted",
        random_state=42,
        n_jobs=-1
    )
    rs.fit(X_train, y_train)
    
    elapsed = time.time() - start_time
    improvement = ((rs.best_score_ - prev_score) / prev_score * 100) if prev_score > 0 else 0
    
    cv_scores_by_iter.append((n, rs.best_score_))
    timing_info.append((n, elapsed))
    
    print(f"{n:<10} {rs.best_score_:<15.4f} {improvement:+.2f}%")
    
    prev_score = rs.best_score_

print("\n" + "="*70)
print("✓ When improvement plateaus, you have enough iterations")
print(f"  In this case: n_iter=50-75 appears sufficient")

# Visualize the plateau
n_values = [x[0] for x in cv_scores_by_iter]
scores = [x[1] for x in cv_scores_by_iter]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Score convergence
ax1.plot(n_values, scores, marker='o', linewidth=2, markersize=8, color='steelblue')
ax1.axhline(y=max(scores), color='red', linestyle='--', alpha=0.5, label='Best Score')
ax1.fill_between(n_values, max(scores) * 0.99, max(scores), alpha=0.2, color='green',
                  label='Plateau Region')
ax1.set_xlabel('n_iter', fontsize=12)
ax1.set_ylabel('Best CV Score', fontsize=12)
ax1.set_title('Finding Optimal n_iter: When Scores Plateau', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Timing
timings = [x[1] for x in timing_info]
ax2.plot(n_values, timings, marker='s', linewidth=2, markersize=8, color='darkorange')
ax2.set_xlabel('n_iter', fontsize=12)
ax2.set_ylabel('Time (seconds)', fontsize=12)
ax2.set_title('computational Cost vs. n_iter', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 6: Visualizing RandomizedSearchCV Results

Unlike GridSearchCV's structured grid, RandomizedSearchCV produces a scattered cloud of results. This reveals the performance landscape.

In [ ]:
# Extract and visualize from our previous RandomizedSearchCV run
results_df_rs = pd.DataFrame(random_search_rf.cv_results_)[[
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_leaf",
    "mean_train_score",
    "mean_test_score",
    "std_test_score"
]]

print("\nTop 10 Configurations from RandomizedSearchCV")
print("="*90)
top_results = results_df_rs.nlargest(10, 'mean_test_score')
print(top_results[['param_n_estimators', 'param_max_depth', 'param_min_samples_leaf', 
                    'mean_test_score', 'std_test_score']].to_string(index=False))

# Visualize the scatter of results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# n_estimators vs. CV score
axes[0, 0].scatter(
    results_df_rs["param_n_estimators"],
    results_df_rs["mean_test_score"],
    alpha=0.6, s=100, c=results_df_rs["mean_test_score"],
    cmap='viridis', edgecolors='black', linewidth=0.5
)
best_idx = results_df_rs["mean_test_score"].idxmax()
best_est = results_df_rs.loc[best_idx, "param_n_estimators"]
best_score = results_df_rs.loc[best_idx, "mean_test_score"]
axes[0, 0].scatter(best_est, best_score, marker='*', s=500, color='red',
                    edgecolors='darkred', linewidth=2, label='Best')
axes[0, 0].set_xlabel("n_estimators", fontsize=11)
axes[0, 0].set_ylabel("CV F1 Score", fontsize=11)
axes[0, 0].set_title("n_estimators vs. CV Performance", fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# max_depth vs. CV score
axes[0, 1].scatter(
    results_df_rs["param_max_depth"],
    results_df_rs["mean_test_score"],
    alpha=0.6, s=100, c=results_df_rs["mean_test_score"],
    cmap='viridis', edgecolors='black', linewidth=0.5
)
best_depth = results_df_rs.loc[best_idx, "param_max_depth"]
axes[0, 1].scatter(best_depth, best_score, marker='*', s=500, color='red',
                    edgecolors='darkred', linewidth=2, label='Best')
axes[0, 1].set_xlabel("max_depth", fontsize=11)
axes[0, 1].set_ylabel("CV F1 Score", fontsize=11)
axes[0, 1].set_title("max_depth vs. CV Performance", fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Train vs. CV scores (overfitting detector)
axes[1, 0].scatter(
    results_df_rs["mean_train_score"],
    results_df_rs["mean_test_score"],
    alpha=0.6, s=100, color='steelblue', edgecolors='black', linewidth=0.5
)
axes[1, 0].plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Perfect (no overfitting)')
axes[1, 0].scatter(results_df_rs.loc[best_idx, "mean_train_score"], best_score,
                   marker='*', s=500, color='red', edgecolors='darkred', linewidth=2, label='Best')
axes[1, 0].set_xlabel("Mean Train Score", fontsize=11)
axes[1, 0].set_ylabel("Mean CV Score", fontsize=11)
axes[1, 0].set_title("Overfitting Detector: Train vs. CV", fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()
axes[1, 0].set_xlim([0.5, 1.0])
axes[1, 0].set_ylim([0.5, 1.0])

# CV score distribution
axes[1, 1].hist(results_df_rs["mean_test_score"], bins=15, color='steelblue',
                alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=best_score, color='red', linestyle='--', linewidth=2,
                   label=f'Best: {best_score:.4f}')
axes[1, 1].axvline(x=results_df_rs["mean_test_score"].mean(), color='green',
                   linestyle='--', linewidth=2, label=f'Mean: {results_df_rs["mean_test_score"].mean():.4f}')
axes[1, 1].set_xlabel("CV F1 Score", fontsize=11)
axes[1, 1].set_ylabel("Frequency", fontsize=11)
axes[1, 1].set_title("Distribution of CV Scores", fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n✓ Scatter plots reveal:")
print("  • Which parameters most strongly affect performance")
print("  • Whether train/CV gap indicates overfitting")
print("  • Score distribution and variability")

## Section 7: GridSearchCV vs. RandomizedSearchCV - When to Use Each

| Dimension | GridSearchCV | RandomizedSearchCV |
|---|---|---|
| Search Type | Exhaustive (all combinations) | Random sampling |
| Scales to high dimensions | ❌ No | ✅ Yes |
| Continuous ranges | Awkward (fixed points only) | Natural (scipy.stats distributions) |
| Guarantees best within search space | ✅ Yes | ❌ No (probabilistic) |
| Specifies exact values | ✅ Yes | ❌ Samples distributions |
| Best for | 2–3 params, narrow ranges, final fine-tuning | 4+ params, wide ranges, initial exploration |
| Speed scaling | Exponential with dimensions | Linear with n_iter |
| Use case | After identifying promising region | Before identifying promising region |

### Decision Flowchart

**Number of hyperparameters ≤ 3?**
- YES: Use GridSearchCV (or RandomizedSearchCV initially, then refine with GridSearchCV)
- NO: Use RandomizedSearchCV

**Range of search space?**
- NARROW (few meaningful values): GridSearchCV
- WIDE (continuous or many discrete values): RandomizedSearchCV

**Time budget?**
- TIGHT: RandomizedSearchCV (n_iter=30–50)
- FLEXIBLE: Hybrid (RandomizedSearchCV exploration + GridSearchCV refinement)

## Section 8: The Hybrid Coarse-to-Fine Tuning Strategy

The most effective workflow combines both methods:

### Phase 1: Broad Exploration (RandomizedSearchCV)

Use **RandomizedSearchCV** to explore a wide search space efficiently and identify which regions look promising.

In [ ]:
print("HYBRID COARSE-TO-FINE TUNING STRATEGY")
print("="*80)

# Phase 1: Broad exploration with RandomizedSearchCV
print("\nPHASE 1: Broad Exploration (RandomizedSearchCV)")
print("-"*80)

param_dist_coarse = {
    "n_estimators": randint(50, 500),
    "max_depth": randint(2, 30),
    "min_samples_leaf": randint(1, 20),
    "max_features": ["sqrt", "log2", None]
}

print("Coarse search space:")
print(f"  n_estimators: [50–500]")
print(f"  max_depth: [2–30]")
print(f"  min_samples_leaf: [1–20]")
print(f"  max_features: [sqrt, log2, None]")
print(f"\nRunning RandomizedSearchCV with n_iter=50...")

rs_coarse = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist_coarse,
    n_iter=50,
    cv=3,
    scoring="f1_weighted",
    random_state=42,
    n_jobs=-1
)

rs_coarse.fit(X_train, y_train)

print(f"\n✓ Phase 1 complete!")
print(f"Best Configuration (Coarse):")
best_params_coarse = rs_coarse.best_params_
for param, value in best_params_coarse.items():
    print(f"  {param}: {value}")
print(f"Best CV F1: {rs_coarse.best_score_:.4f}")

# Identify promising regions
est_min, est_max = best_params_coarse['n_estimators'] - 50, best_params_coarse['n_estimators'] + 50
depth_min, depth_max = max(2, best_params_coarse['max_depth'] - 3), best_params_coarse['max_depth'] + 3
leaf_min, leaf_max = max(1, best_params_coarse['min_samples_leaf'] - 5), best_params_coarse['min_samples_leaf'] + 5

print(f"\nIdentified promising regions:")
print(f"  n_estimators: [{est_min}–{est_max}]")
print(f"  max_depth: [{depth_min}–{depth_max}]")
print(f"  min_samples_leaf: [{leaf_min}–{leaf_max}]")

# Phase 2: Precise refinement with GridSearchCV
print("\n" + "="*80)
print("PHASE 2: Precise Refinement (GridSearchCV)")
print("-"*80)

param_grid_fine = {
    "n_estimators": list(range(max(50, est_min), min(500, est_max) + 1, 10)),
    "max_depth": list(range(depth_min, depth_max + 1)),
    "min_samples_leaf": list(range(leaf_min, leaf_max + 1))
}

print(f"Fine search space:")
print(f"  n_estimators: {param_grid_fine['n_estimators']}")
print(f"  max_depth: {param_grid_fine['max_depth']}")
print(f"  min_samples_leaf: {param_grid_fine['min_samples_leaf']}")

n_grid_combos = len(param_grid_fine['n_estimators']) * len(param_grid_fine['max_depth']) * len(param_grid_fine['min_samples_leaf'])
print(f"\nGrid size: {n_grid_combos} combinations × 3-fold CV = {n_grid_combos * 3} model fits")
print(f"Running GridSearchCV...")

gs_fine = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid_fine,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1
)

gs_fine.fit(X_train, y_train)

print(f"\n✓ Phase 2 complete!")
print(f"Best Configuration (Fine):")
for param, value in gs_fine.best_params_.items():
    print(f"  {param}: {value}")
print(f"Best CV F1: {gs_fine.best_score_:.4f}")

# Phase 3: Test evaluation
print("\n" + "="*80)
print("PHASE 3: Test Evaluation")
print("-"*80)

y_pred_hybrid = gs_fine.best_estimator_.predict(X_test)
test_f1_hybrid = f1_score(y_test, y_pred_hybrid, average='weighted')

print(f"Final Test F1 Score: {test_f1_hybrid:.4f}")
print(f"\n✓ Hybrid approach typically finds comparable or better solutions")
print(f"  with 20–30% of the compute time of exhaustive GridSearchCV")

## Section 9: Data Leakage and Best Practices

### The Critical Rule

**Never let test-set information influence hyperparameter selection.**

### Correct Workflow

```
All Data
┌─────────────────────────────────────┐
│ Training Set        │ Test Set      │
│                     │               │
│ RandomizedSearchCV  │ Evaluated     │
│ runs here           │ once here     │
│ (CV on train only)  │ (after tuning)|
└─────────────────────────────────────┘
```

### Critical: Use Pipelines

**Wrong:** Scaling outside the RandomizedSearchCV loop
```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
random_search.fit(X_train_scaled, y_train)  # CV is contaminated!
```

**Correct:** Preprocessing inside the pipeline
```python
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier())
])
random_search.fit(X_train, y_train)  # Clean: scaler refitted each fold
```

### Practical Checklist Before Finalizing
- ✓ Train/test split done BEFORE any tuning
- ✓ Preprocessing inside a pipeline
- ✓ `random_state` set for reproducibility
- ✓ `n_iter` appropriate for search space complexity
- ✓ `return_train_score=True` to inspect overfitting
- ✓ Scoring metric aligned with business objective
- ✓ Test set evaluated EXACTLY ONCE after tuning
- ✓ Full results table reviewed and interpreted
- ✓ Improvement over baseline documented

## Section 10: Common Mistakes to Avoid

1. **Too Few Iterations**
   - With n_iter=10 over 5 hyperparameters, you've only sampled ~0.001% of a discretized space
   - Use at least n_iter=50 for exploratory work; 100+ for final tuning

2. **Not Setting random_state**
   - Every run samples different configurations → non-reproducible results
   - Always set `random_state` for reproducibility

3. **Using uniform() for Regularization Parameters**
   - `uniform(0.0001, 100)` places almost all probability mass > 1
   - Use `loguniform(1e-4, 1e2)` for parameters spanning multiple orders of magnitude

4. **Not Using Pipelines with Scaling**
   - Causes CV-level leakage and inflates scores
   - Always wrap preprocessing inside the pipeline

5. **Ignoring the Train/CV Gap**
   - With `return_train_score=True`, inspect whether best config is overfitting
   - Train accuracy 99%, CV accuracy 75% = overfitting detected

6. **Comparing Inconsistent CV Strategies**
   - Model A tuned with 3-fold CV is not comparable to Model B with 5-fold CV
   - Always use the same CV strategy across models

## Section 11: Reporting Results

A complete, transparent RandomizedSearchCV report includes:

```
Model:              Random Forest Classifier
Search Method:      RandomizedSearchCV
Parameter Space:    5 dimensions (wide ranges)
Iterations (n_iter): 50
CV Strategy:        5-fold stratified
Scoring Metric:     F1 (weighted)
Random State:       42 (reproducible)

Baseline F1:           0.55  (majority-class predictor)
Untuned RF F1:         0.73  (default sklearn parameters)
RandomizedSearch CV F1: 0.82 ± 0.03  (mean ± std from CV)
Final Test F1:         0.80  (evaluated exactly once)

Best Configuration:
  n_estimators:     247
  max_depth:        8
  min_samples_leaf: 4
  max_features:     sqrt

Improvement from tuning: +0.07 F1 (9.6% relative improvement)
Hybrid refinement with GridSearchCV: Applied to top 10 configs, no improvement found
```

Always include:
- The `random_state` value
- The number of iterations used
- CV mean ± std (not just best score)
- Final test score (evaluated exactly once, after tuning)
- Baseline and untuned performance
- Best hyperparameter values for reproducibility

## Closing Reflection

RandomizedSearchCV brings **scalability** to hyperparameter tuning. It acknowledges a fundamental truth: exhaustive search is unnecessary because most hyperparameters have limited impact, and the ones that matter can be explored efficiently with random sampling.

It enables you to search over **continuous distributions** — finding the value 7.3 where grid search was stuck between 5 and 10. It scales **linearly with n_iter** rather than exponentially with dimensions. And with `n_jobs=-1`, it parallelizes across every available CPU core at no additional cost.

**Used alone:** Handles broad exploration efficiently.  
**Used with GridSearchCV:** The hybrid coarse-to-fine strategy finds near-optimal configurations faster than either method alone.  
**Always with pipelines:** Prevents leakage and ensures reproducible, trustworthy results.

**Optimization is about intelligent exploration—not brute force.**